In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

## Total Returns Extended to Commodities

### Data sources
- st louis 3m tbill used as risk free rate for determination of sharpe, it can be used as approximation funding cost available since 1930
- shiller 10y bond rate and equity return and div, this allows computing 10y total return and equity total return
- spot commo prices from CMO, spot can be used approx for return on precious metals, and possibly copper. Others, notable oil spot price return is missing massive future roll return
- CL1 and CL6 historical data from bloomberg, since 1991. This allows computing future roll return, which is significant for oil.

In [ ]:
def tbillrate():
    df = pd.read_csv("data/TB3MS.csv")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["TB3MS"] /= 100
    return df.set_index("observation_date")
def compute_gold_tr(df):
    for colname in df.columns:
        df[colname+'TR'] = df[colname]/df[colname].shift(1)
    return df
def goldprice(col):
    df = pd.read_csv("data/cmo-data-monthly.csv")
    df = df[["date"]+list(col.keys())]
    df["date"] = pd.to_datetime(df["date"])+dt.timedelta(days=1)
    df = df.loc[df["date"]>=dt.datetime(1971,2,1)]
    return compute_gold_tr(df.set_index("date").rename(columns=col)).copy()
def read_shiller_out():
    df = pd.read_csv("data/shiller_out.csv")
    df['Date'] = pd.to_datetime(df['Date'], format="%Y-%m-%d")+dt.timedelta(days=-14) # imported as 14th, will be used as 1st 
    df = df.set_index("Date")
    df = df.join(tbillrate(),how="inner")
    compute_tbill_tr(df)
    return df.copy()
def read_shiller_cmo(col):
    return read_shiller_out().join(goldprice(col),how="inner").copy()
          
"""
def compute_bond_tr(df):
    r = df['Rate10y']
    T = 10
    duration = -(1-np.exp(-r*T))/r
    bondcarry = r.shift(1)/12
    bondtotalret = 1+(r-r.shift(1))*duration+bondcarry
    df["bondTR"] = bondtotalret
def compute_eq_tr(df):
    df['eqTR'] = df['SP500']/df['SP500'].shift(1)+df['Div']/12/df['SP500']
def compute_cpi_tr(df):
    df['cpiTR'] = 1+df['CPI'].pct_change()
    """
def compute_tbill_tr(df):
    df['tbTR'] = 1+df['TB3MS'].shift(1)*365/360/12 # compute with shift with last rate, as if this was a 1m rate

def metrics_monthly_ret(df,cols):
    """
    Estimates constant excess return mu* using discounted log-returns
    S_t / M_t, allowing the risk-free rate r_t to vary over time.
    """
    df = df[cols].copy()
    trcolumns = [c for c in df.columns if "TR"==c[-2:] and c not in ["cpiTR","portTR"]]
    logret = np.log(df[trcolumns])          # monthly log returns
    # --- Discounted log-returns: m*_t = ln(S_t) - ln(M_t) ---
    # logret['tbTR'] = ln(M_t) - ln(M_{t-1}) = r_t * dt
    discounted_logret = logret.sub(logret['tbTR'], axis=0)  # m*_t for ALL assets
    # Drop tbTR itself (its discounted return is identically zero)
    riskycol = [c for c in trcolumns if c !='tbTR']
    discounted_logret = discounted_logret[riskycol]
    # --- Estimators on discounted series ---
    mstar_bar = discounted_logret[riskycol].mean()* 12  # annualize mean of m*_t, drift of log disc asseet 
    sigma     = discounted_logret[riskycol].std()* np.sqrt(12)   # annualized volatility
    # mu* dt = E[d ln S/M] + 0.5 * Var[ln S/M]  
    mustar = mstar_bar + 0.5 * sigma**2 
    # Average risk-free rate for reporting
    r_bar = logret['tbTR'].mean()*12                      # annualized average r_t
    # Risk premium in discrete compounding
    mustar_discrete = np.exp(mustar) - 1
    # --- Same post-processing as before ---
    data = {'mustar': mustar_discrete, 'sigma': sigma}
    data['sharpe'] = data['mustar'] / data['sigma']
    corr = discounted_logret[riskycol].corr()
    cov = discounted_logret[riskycol].cov() * 12
    wstar = np.linalg.solve(cov, data['mustar'])
    K = np.abs(np.sum(wstar))
    data['w'] = w = wstar / K
    for c in riskycol:
        data[f'rho({c[:-2]})'] = corr[c]
    metricsdf = pd.DataFrame(data, index=riskycol)
    haslong = [1 if "Carry" not in c else 0 for c in riskycol]           
    hascarry = [1 if "Carry" in c else 0 for c in riskycol] 
    if len(riskycol)!=2 or np.sum(hascarry)==0 or np.sum(haslong)==0:
        print(riskycol)
        port_ret       = df.loc[discounted_logret.index,"tbTR"]+np.dot(np.expm1(discounted_logret), w)
        df.loc[discounted_logret.index, "portTR"]   = port_ret
        port_logret    = np.log(port_ret)
        port_mstar_bar = port_logret.mean()* 12  
        port_sigma     = port_logret.std()* np.sqrt(12) 
        port_mustar    = port_mstar_bar + 0.5 * port_sigma**2 
        port_mustar_discrete = np.exp(port_mustar) - 1
        metricsdf = pd.concat([metricsdf,pd.DataFrame({
            'mustar': [port_mustar_discrete],
            'sigma': [port_sigma],
            'sharpe': [port_mustar_discrete / port_sigma],
            'w': [np.sum(w)]
        }, index=['portTR'])])
    return metricsdf, r_bar, K, df

def getcolor(c):
    if "eq" in c:
        color = "blue"
    elif "bond" in c:
        color = "darkgreen"
    elif "oil" in  c:
        color = "black"
    elif "copper" in c:
        color = "darkorange"
    elif "gold" in c:
        color = "gold"
    elif "silver" in c:
        color = "silver"
    elif "platinum" in c:
        color = "lightgray"
    elif "port" in c:
        color = "purple"
    return color

def show_returns(df,filename,excl):
    cols = [c for c in df.columns if c[-2:]=="TR" and c not in excl]
    dfmetrics,r,K,df = metrics_monthly_ret(df,cols)
    print(dfmetrics.index)
    print(dfmetrics)
    for c in dfmetrics.index:
        ls = "--" if "Carry" in c else "-"
        plt.plot(np.cumprod(df[c]/df["tbTR"]),color=getcolor(c), linestyle=ls,
                label=f"{c[:-2]}: $\mu^*$={dfmetrics.loc[c,'mustar']:.1%}, $\sigma$={dfmetrics.loc[c,'sigma']:.0%}, S={dfmetrics.loc[c,'sharpe']:.2f}, w={dfmetrics.loc[c,'w']:.0%}")
    plt.legend()
    plt.title(f"Asset Total Return r={r:.1%} K={K:.1f}")
    plt.ylabel("log total return")
    plt.yscale('log')
    datesstr = f"from {str(df.index[0])[:10]} to {str(df.index[-1])[:10]}"
    plt.xlabel(datesstr)
    print(datesstr)
    print(f"r={r:.2%} K={K:.1f} (kelly leverage)")
    plt.grid(True)  
    if not filename is None:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()
    return dfmetrics

def show_all_returns(df,show,excl):
    assets = [c[:-2] for c in df.columns if "TR" in c and c not in ["cpiTR","tbTR","portTR"]]
    filename = None if show else "_".join(assets).replace(" (spot)","spot")+".png"
    dfmetrics = show_returns(df,filename,excl)
    return dfmetrics,filename


### Method
- tbill total return $\mu_r= 1+r . 365/360/12$ where $r$ is the rate in ACT.360 convention
- equity total return $\mu_e=P(i+1)/P(i) + D(i)/12$, where $P$ is price, $D$ annual dividend
- bond total return $\mu_b=A(i+1) (y(i+1)-y(i)) + y(i) / 12$ where $A$ is the annuity, $y$ the 10y yield
- commo price return $\mu_o=S(i+1)/S(i)$ where $S$ is spot price (this is total return for precious metals)
- contango amount $c=(F(i+6)-F(i+1))/F(i+6)$, contango rate $y_c=c^{1/5}$
- commo future total return $\mu_o=S(i+1)/S(i)-y_c$ 

### CMO Data
- crude oil spot shows big supply shocks in the 70s, return is significant
- gold, platinum, and silver are similar but gold has the lowest vol, 
- copper only recently started to beat inflation  from the 2000s (energiewende or China demand growing)
- softs tend to grow only 2%-3%, so return is flat above inflation, no excess return.
- algo prefers gold and copper to silver and platinum. The latter seems to be within efficient frontier of gold and copper.

### Metrics and Optimal Portfolio
- monthly log ret annualized vol $\sigma$
- monthly annualized expected return $\mu$ 
- monthly annualized expected excess return $\mu^*=\mu-r$
- Sharpe = $\mu^*/\sigma$
- optimal weightws $w^* = \Sigma^{-1} \mu^*$
- Optimal Kelly leverage $K=\sum(w)$
- normed weights $w = w^*/K$

## Shiller +Fed Data Only: Equity and Bond excess return from 1934 (92 years)

In [ ]:
dfsh = read_shiller_out()
show_all_returns(dfsh,True,[])



### Rate Strategy

In [ ]:


x = dfsh["TB3MS"]
y = dfsh["Rate10y"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3M tbill rate")
plt.ylabel("10y rate")
plt.title("10y rate vs 3m rate")
plt.show()
x = (dfsh["TB3MS"]).shift(1)
y = dfsh["bondTR"]-dfsh["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3m rate")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs 3m rate - pivot:{-b/a:.2%}")
plt.show()
print(a,b,errstd)
x = (dfsh["Rate10y"]-dfsh["TB3MS"]).shift(1)
y = dfsh["bondTR"]-dfsh["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("10y - 3m rate spread")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)


In [ ]:
def regress_asset(dfsh,asset):
    x = (dfsh[asset+"TR"]-dfsh["tbTR"]).rolling(1).mean().shift(1)
    y = dfsh[asset+"TR"]-dfsh["tbTR"]
    return x,y
def carrystrat_asset(dfsh,x,y,asset,pivot):
    dfsh[asset+"CarryTR"] = dfsh["tbTR"]+y*np.where(x > pivot, 1,-1)
    return dfsh 
x,y = regress_asset(dfsh,"bond")
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfsh = carrystrat_asset(dfsh,x,y,"bond",pivot=-0.002)
show_all_returns(dfsh,True,["eqCarryTR","eqTR"])


### Equity Strategy: momentum

In [ ]:

x = (dfsh["TB3MS"]).shift(1)
y = dfsh["eqTR"]-dfsh["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3m rate")
plt.ylabel("eq net total return")
plt.title(f"eq net return vs 3m rate - pivot:{-b/a:.2%}")
plt.show()
print(a,b,errstd)
x = (dfsh["Rate10y"]-dfsh["TB3MS"]).shift(1)
y = dfsh["eqTR"]-dfsh["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("10y - 3m rate spread")
plt.ylabel("eq net total return")
plt.title(f"eq net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)


In [ ]:

x,y = regress_asset(dfsh,"eq")
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("eq net total return")
plt.title(f"eq net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfsh = carrystrat_asset(dfsh,x,y,"eq",pivot=-0.02)
show_all_returns(dfsh,True,["bondCarryTR","bondTR"])


In [ ]:
show_all_returns(dfsh,True,["eqTR","bondTR"])


## Shiller + Fed + CMO Data: Gold since 1971 (65 years of data)

In [ ]:
col = {"CRUDE_DUBAI":"oil (spot)"}
for c in ["GOLD","COPPER"]:
    col[c] = c.lower()
dfcmo = read_shiller_cmo(col)
dfmetrics,filename = show_all_returns(dfcmo,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
col = {"CRUDE_DUBAI":"oil (spot)"}
for c in ["GOLD","COPPER","PLATINUM","SILVER"]:
    col[c] = c.lower()
dfcmo = read_shiller_cmo(col)
dfmetrics,filename = show_all_returns(dfcmo,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
asset = "gold"
dfcmo = read_shiller_cmo({asset.upper():asset})
x,y = regress_asset(dfcmo,asset)
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("{asset} net total return")
plt.title(f"{asset} net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfsh = carrystrat_asset(dfcmo,x,y,asset,pivot=-0.012)
show_all_returns(dfcmo,True,["eqTR","bondTR"])


In [ ]:
asset = "silver"
dfcmo = read_shiller_cmo({asset.upper():asset})
x,y = regress_asset(dfcmo,asset)
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("{asset} net total return")
plt.title(f"{asset} net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfsh = carrystrat_asset(dfcmo,x,y,asset,pivot=-0.01)
show_all_returns(dfcmo,True,["eqTR","bondTR"])


In [ ]:
asset = "platinum"
dfcmo = read_shiller_cmo({asset.upper():asset})
x,y = regress_asset(dfcmo,asset)
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("{asset} net total return")
plt.title(f"{asset} net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfsh = carrystrat_asset(dfcmo,x,y,asset,pivot=-0.01)
show_all_returns(dfcmo,True,["eqTR","bondTR"])


## Shiller + Fed + CMO + Bloomberg Data: from 1991 (35y)

In [ ]:
def getbbdata(tick):
    dfcl = pd.read_csv(f"data/{tick.lower()}.csv")
    dfcl["date"] = pd.to_datetime(dfcl["date"])
    dfcl = dfcl.set_index("date")
    dfcl = dfcl.resample('M').last()
    dfcl = dfcl.reset_index()
    dfcl["date"] = dfcl["date"]+dt.timedelta(days=1)
    dfcl = dfcl.set_index("date")
    # Primary axis: CL1 and CL6
    fig, ax1 = plt.subplots(figsize=(10, 6))
    color = "black" if tick=="CL" else "darkorange"
    ax1.plot(dfcl[tick+"1"], label=tick+"1", color=color)
    ax1.plot(dfcl[tick+"6"], label=tick+"6", color=color, linestyle="--")
    ax1.set_ylabel("price")
    ax1.tick_params(axis='y', labelcolor=color)
    # Secondary axis: CL6 - CL1 (in gray)
    ax2 = ax1.twinx()
    ax2.plot(1 - dfcl[tick+"1"]/dfcl[tick+"6"], label=f"1 - {tick}1/{tick}6", color="gray", linestyle="--", alpha=0.8)
    ax2.set_ylabel("contango", color="gray")
    ax2.axhline(y=0,color="gray")
    ax2.tick_params(axis='y', labelcolor="gray")
    # Legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    plt.title(f"{tick} Price and Contango")
    plt.grid(True, alpha=0.3)
    plt.show()
    return dfcl

def addcontango(dfcl,tick):
    contango = (1+(dfcl[tick+"6"]-dfcl[tick+"1"])/dfcl[tick+"6"])**(1/5)-1-dfcl["TB3MS"]/12 # net carry
    totret   = dfcl[tick+"6"].pct_change()-contango.shift(1)+1#+(dfcl["tbTR"]-1) # excess return mu*_o=mu_o-mu_
    dfcl[f"{tickmap[tick]}CarryTR"] = dfcl["tbTR"]+(totret-1)*np.where(contango<0.0,1,-1) 
    dfcl[f"{tickmap[tick]}TR"] = totret
    dfcl[tick+"contango"] = contango
    return dfcl
tickmap = {"CL":"oil","HG":"copper"}
tick="CL"
dfcl = getbbdata(tick).join(getbbdata(tick="HG"))
precious = ["gold","silver","platinum"]
col = {}
for c in precious:
    col[c.upper()] = c
df = read_shiller_cmo(col)
dfcl = dfcl.join(df)

dfcl = addcontango(dfcl,tick)
dfcl = addcontango(dfcl,tick="HG")


## Future Strats based on contango

### Oil Contango based Strategy
- most time is backwardation
- backwardation appears bullish, contango is bearish
- oil total return shows future total return, oil carry strategy goes long in backwardation, short in contango

In [ ]:
def contangocarrypnl(dfcl,tick):
    x = (dfcl[tick+"contango"]).shift(1)
    y = dfcl[f"{tickmap[tick]}TR"]-1
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = x[mask].values
    y_clean = y[mask].values
    coeffs = np.polyfit(x_clean, y_clean, 1)
    a, b = coeffs
    y_pred = a * x_clean + b
    errstd = np.std(y_clean - y_pred)
    plt.scatter(x, y, alpha=0.2)
    plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
    plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
             transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
    plt.axvline(x=-b/a,color="black")
    plt.xlabel(f"{tickmap[tick]} contango")
    plt.ylabel(f"6M {tickmap[tick]} future total return")
    plt.title(f"{tickmap[tick]} total return vs contango - pivot: {-b/a:.2%}")
    plt.show()
    print(a,b,errstd)
    color = "black" if tick=="CL" else "darkorange"
    plt.plot(np.cumprod(dfcl[f"{tickmap[tick]}CarryTR"]),label=f"{tickmap[tick]} carry strat",color=color)
    plt.plot(np.cumprod(dfcl[f"{tickmap[tick]}TR"]),label=f"long {tickmap[tick]} future total return", linestyle="--",color=color)
    plt.yscale('log')
    plt.legend()
    plt.grid(True)

In [ ]:
contangocarrypnl(dfcl,tick="CL")

In [ ]:
dfcl

In [ ]:
contangocarrypnl(dfcl,tick="HG")

In [ ]:
x,y = regress_asset(dfcl,"eq")
dfcl = carrystrat_asset(dfcl,x,y,"eq",-0.02)
x,y = regress_asset(dfcl,"bond")
dfcl = carrystrat_asset(dfcl,x,y,"bond",-0.002)
for c in precious:
    x,y = regress_asset(dfcl,c)
    dfcl = carrystrat_asset(dfcl,x,y,c,-0.01)
excl=["oilTR","copperTR","eqTR","bondTR","goldTR","silverTR","platinumTR"]
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=excl)
print(dfmetrics.to_markdown(floatfmt=".2%"))



In [ ]:
excl += ["silverCarryTR","platinumCarryTR"]
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=excl)
print(dfmetrics.to_markdown(floatfmt=".2%"))

## Future roll impact on pnl for oil
- data from 1991 only
- impact of backwardation is massively positive
- be careful with oil front contract, it gets negative in mar 2020
- we can use oil future total return, or much better, oil carry strategy

### Rate study
- 10y bond is considered risky even though gov can print money because it bears rate risk
- 3m bond is considered risk free
- 10y vs 3m scatter plot shows correlation
- bond net return is flat vs 3m rate
- bond net return has slop vs 10y03m spread

## Gold return
- gold does well when inflation fear is high: when bonds don't return much over tbills.

## Copper has econ PhD
- copper move are highly correlated to past 3m equity return

## Eq future return
- appears to be strongly correlated to prev 8m perf

In [ ]:
excl += ["copperCarryTR"]
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=excl)
print(dfmetrics.to_markdown(floatfmt=".2%"))

In [ ]:
excl = [c for c in dfcl.columns if c[-2:]=="TR" and "Carry" in c]+["silver","platinum"]
dfmetrics,filename = show_all_returns(dfcl,show=True,excl=excl)
print(dfmetrics.to_markdown(floatfmt=".2%"))

In [ ]:
asset = "oil"
x,y = regress_asset(dfcl,asset)
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("{asset} net total return")
plt.title(f"{asset} net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfcl = carrystrat_asset(dfcl,x,y,asset,pivot=-0.03)
excl = [c for c in dfcl.columns if c not in ["oilTR", "oilCarryTR", "tbTR"]]
show_all_returns(dfcl,True,excl)


In [ ]:
asset = "copper"
x,y = regress_asset(dfcl,asset)
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("prev return")
plt.ylabel("{asset} net total return")
plt.title(f"{asset} net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)
print(dfsh.columns)
dfcl = carrystrat_asset(dfcl,x,y,asset,pivot=-0.06)
excl = [c for c in dfcl.columns if c not in ["copperTR", "copperCarryTR", "tbTR"]]
show_all_returns(dfcl,True,excl)
